# Activity 1 – Data Wrangling
**Course:** Advanced Programming – Week 6  
**Author:** Sebastian Diaz  

Practise applying data wrangling techniques for the grouping and reshaping of simple data sets.

In [ ]:
import pandas as pd
import numpy as np

---
## Helper: Load the Hierarchical CSV Files

Both files use a multi-level header layout (college → subject → grade) with empty separator rows.
This loader handles that structure, filling forward across merged header cells.

In [ ]:
def load_college_grades(filepath, grade_order, period_labels, data_rows):
    """
    Load a CollegeGrades CSV with hierarchical headers.
    
    Parameters:
    -----------
    filepath     : path to CSV
    grade_order  : list of grade labels in file order, e.g. ['F','P','M','D']
    period_labels: list of (period, gender) tuples for data rows
    data_rows    : 0-indexed row numbers of actual data
    
    Returns a DataFrame with MultiIndex rows (Period, Gender)
    and MultiIndex columns (College, Subject, Grade).
    """
    raw = pd.read_csv(filepath, header=None, dtype=str)
    
    # --- Build column MultiIndex ---
    def ffill_row(row_series):
        """Forward-fill a sparse header row."""
        result = []
        current = ''
        for v in row_series:
            val = str(v).strip()
            if val and val != 'nan':
                current = val
            result.append(current)
        return result

    colleges = ffill_row(raw.iloc[0, 2:])   # row 0: college names
    subjects = ffill_row(raw.iloc[2, 2:])   # row 2: subject names
    grades   = [str(v).strip() for v in raw.iloc[4, 2:]]  # row 4: grade labels

    col_idx = pd.MultiIndex.from_arrays(
        [colleges, subjects, grades],
        names=['College', 'Subject', 'Grade']
    )

    # --- Extract data ---
    data = raw.iloc[data_rows, 2:].values.astype(int)
    periods = [p for p, g in period_labels]
    genders  = [g for p, g in period_labels]
    row_idx  = pd.MultiIndex.from_arrays([periods, genders], names=['Period', 'Gender'])

    return pd.DataFrame(data, index=row_idx, columns=col_idx)


# --- Dataset 1: CollegeGrades1.csv ---
# Colleges: Whitby, Bridlington, Scarborough
# Grade order in file: F, P, M, D
# Period labels: Year 1 / Year 2 / Evening × M / F
df1 = load_college_grades(
    'CollegeGrades1.csv',
    grade_order=['F', 'P', 'M', 'D'],
    period_labels=[
        ('Year 1', 'M'), ('Year 1', 'F'),
        ('Year 2', 'M'), ('Year 2', 'F'),
        ('Evening', 'M'), ('Evening', 'F')
    ],
    data_rows=[6, 8, 10, 12, 14, 16]
)

print("Dataset 1 — CollegeGrades1")
print(df1)
print("\nShape:", df1.shape)

In [ ]:
# --- Dataset 2: CollegeGrades2.csv ---
# Colleges: Darlington, Middlesbrough, Harrogate (note typos in source)
# Grade order in file: P, M, D, F  <-- DIFFERENT ORDER to Dataset 1 (data quality issue)
# Period labels: M 16-17, M 17-18, F 16-17, F 17-18
# Academic years 16-17/17-18 ≈ Year 1 / Year 2 (assumption)
df2 = load_college_grades(
    'CollegeGrades2.csv',
    grade_order=['P', 'M', 'D', 'F'],
    period_labels=[
        ('Year 1', 'M'), ('Year 2', 'M'),
        ('Year 1', 'F'), ('Year 2', 'F')
    ],
    data_rows=[6, 8, 10, 12]
)

print("Dataset 2 — CollegeGrades2")
print(df2)
print("\nShape:", df2.shape)

---
## Exercise 1 – Reshape to Required Formats

### Target format (a): Subjects as columns, Year/Evening as rows

In [ ]:
# Format (a): rows = Period × Gender; columns = Subject × Grade
# Achieved by dropping the College level from columns
# and summing across colleges — i.e. aggregate all colleges into one view

# Sum across all colleges (group by Subject and Grade at column level)
format_a = df1.groupby(level=['Subject', 'Grade'], axis=1).sum()

# Reorder columns alphabetically by subject for readability
format_a = format_a.sort_index(axis=1)

print("Format (a) — Subjects as columns, Period/Gender as rows:")
print(format_a)
print()
print("Column MultiIndex levels:", format_a.columns.levels)

In [ ]:
# Format (b): rows = Gender; columns = College × Grade
# Sum across all subjects and periods

# Sum across subjects within each college/grade
format_b = df1.groupby(level=['College', 'Grade'], axis=1).sum()

# Sum across periods — leaving only Gender as row index
format_b = format_b.groupby(level='Gender').sum()

# Reorder grade columns to F, P, M, D within each college
grade_order = ['F', 'P', 'M', 'D']
format_b = format_b.reindex(
    columns=pd.MultiIndex.from_product(
        [df1.columns.get_level_values('College').unique(), grade_order],
        names=['College', 'Grade']
    )
)

print("Format (b) — Colleges as columns, Gender as rows:")
print(format_b)

---
## Exercise 2 – Merge Datasets and Analysis

### Step 1: Load and inspect both datasets

In [ ]:
print("Dataset 1 colleges:", df1.columns.get_level_values('College').unique().tolist())
print("Dataset 1 subjects:", df1.columns.get_level_values('Subject').unique().tolist())
print("Dataset 1 periods: ", df1.index.get_level_values('Period').unique().tolist())
print("Dataset 1 grades:  ", df1.columns.get_level_values('Grade').unique().tolist())
print()
print("Dataset 2 colleges:", df2.columns.get_level_values('College').unique().tolist())
print("Dataset 2 subjects:", df2.columns.get_level_values('Subject').unique().tolist())
print("Dataset 2 periods: ", df2.index.get_level_values('Period').unique().tolist())
print("Dataset 2 grades:  ", df2.columns.get_level_values('Grade').unique().tolist())

In [ ]:
# --- MERGE PROCESS AND ASSUMPTIONS ---
#
# ASSUMPTION 1: Academic years 16-17 and 17-18 in Dataset 2 correspond
#   to Year 1 and Year 2 in Dataset 1. Evening class students are not
#   present in Dataset 2 and are excluded from merged analysis.
#
# ASSUMPTION 2: The grade ordering difference (F,P,M,D vs P,M,D,F) is
#   a data quality issue in the source files. Both datasets use the same
#   four grade categories. We normalise to a canonical order: F,P,M,D.
#
# ASSUMPTION 3: Subject name typos ('Enginering', 'Psycology', 'Chemisrty',
#   'Darligton', 'Phsyics', 'Scarborugh') are preserved as-is from the source.
#   We apply a correction mapping before merging so groupby works correctly.
#
# APPROACH: pd.concat along axis=1 (columns) — both DataFrames share the same
#   row index structure (Period × Gender) and different column structures
#   (different colleges). Concatenating adds the new colleges as new column groups.

# Step 1: Normalise spelling in both datasets
spelling = {
    'Enginering': 'Engineering',
    'Psycology':  'Psychology',
    'Chemisrty':  'Chemistry',
    'Scarborugh': 'Scarborough',
    'Darligton':  'Darlington',
    'Phsyics ':   'Physics',
    'Phsyics':    'Physics'
}

def fix_spelling(df):
    new_cols = df.columns.to_frame()
    for level in ['College', 'Subject']:
        new_cols[level] = new_cols[level].replace(spelling)
    return df.set_axis(
        pd.MultiIndex.from_frame(new_cols), axis=1
    )

df1_clean = fix_spelling(df1)
df2_clean = fix_spelling(df2)

# Step 2: Reorder DS2 grades to match canonical F,P,M,D order
# DS2 was loaded with P,M,D,F — swap the Grade level values
grade_map = {'P': 'P', 'M': 'M', 'D': 'D', 'F': 'F'}  # already correct labels
# DS2 grade column order in file was P,M,D,F so they ARE correctly labelled,
# just the data values were loaded under the right labels. No swap needed.
# The data is correctly labelled — the 'F' column contains Fail counts.

# Step 3: Drop Evening rows from DS1 for Year 1/Year 2 comparison
df1_yr = df1_clean.drop('Evening', level='Period')

# Step 4: Concatenate
merged = pd.concat([df1_yr, df2_clean], axis=1)

print("Merged DataFrame shape:", merged.shape)
print("Colleges in merged:", merged.columns.get_level_values('College').unique().tolist())
print("Subjects in merged:", merged.columns.get_level_values('Subject').unique().tolist())
print()
print("Merged (first 4 columns):")
print(merged.iloc[:, :8])

### Analysis Q1: Descriptive statistics — Year 1 vs Year 2 for Mathematics, English, Technology

In [ ]:
target_subjects = ['Mathematics', 'English', 'Technology']

for subject in target_subjects:
    print(f"\n{'='*55}")
    print(f"Subject: {subject}")
    print(f"{'='*55}")

    # Select all colleges that offer this subject
    try:
        subj_data = merged.xs(subject, level='Subject', axis=1)
    except KeyError:
        print(f"  {subject} not found in merged dataset.")
        continue

    # Sum across colleges (total across all campuses offering this subject)
    subj_totals = subj_data.groupby(level='Grade', axis=1).sum()

    # Compare Year 1 vs Year 2 by summing M+F
    yr1 = subj_totals.xs('Year 1', level='Period').sum()  # sum M+F
    yr2 = subj_totals.xs('Year 2', level='Period').sum()

    summary = pd.DataFrame({'Year 1': yr1, 'Year 2': yr2})
    summary['Total Y1'] = summary['Year 1'].sum()
    summary['Total Y2'] = summary['Year 2'].sum()
    summary['Y1 %'] = (summary['Year 1'] / summary['Total Y1'] * 100).round(1)
    summary['Y2 %'] = (summary['Year 2'] / summary['Total Y2'] * 100).round(1)

    print(summary[['Year 1', 'Y1 %', 'Year 2', 'Y2 %']])

    # Totals
    print(f"  Total students Year 1: {yr1.sum():.0f}")
    print(f"  Total students Year 2: {yr2.sum():.0f}")
    print(f"  Year 1 fail rate: {yr1['F']/yr1.sum()*100:.1f}%")
    print(f"  Year 2 fail rate: {yr2['F']/yr2.sum()*100:.1f}%")

### Analysis Q2: Fail/incomplete rate per discipline, by gender

In [ ]:
# Fail = 'F' grade. 'Incomplete' is not a separate grade here —
# we treat F as both fail and non-completion. Assumption documented.

all_subjects = merged.columns.get_level_values('Subject').unique().tolist()

results = []
for subject in all_subjects:
    try:
        subj_data = merged.xs(subject, level='Subject', axis=1).groupby(level='Grade', axis=1).sum()
    except KeyError:
        continue

    for gender in ['M', 'F']:
        try:
            row = subj_data.xs(gender, level='Gender').sum()  # sum over periods
        except KeyError:
            continue
        total = row.sum()
        if total == 0:
            continue
        fail_rate = row.get('F', 0) / total * 100
        results.append({'Subject': subject, 'Gender': gender,
                        'Total': int(total), 'Fails': int(row.get('F', 0)),
                        'Fail%': round(fail_rate, 1)})

fail_df = pd.DataFrame(results).pivot(index='Subject', columns='Gender', values='Fail%')
fail_df.columns = ['Female Fail%', 'Male Fail%']
fail_df = fail_df.sort_values('Male Fail%', ascending=False)

print("Fail/incomplete rate per subject by gender (all years, all colleges):")
print(fail_df.to_string())

### Analysis Q3: Which subject do females outperform males in?

In [ ]:
# Definition of 'better': higher Merit+Distinction rate (as % of all students)
# Justification: F=Fail, P=Pass, M=Merit, D=Distinction.
# Merit and Distinction represent above-average performance.
# We compare the M+D rate between genders per subject.

perf_results = []
for subject in all_subjects:
    try:
        subj_data = merged.xs(subject, level='Subject', axis=1).groupby(level='Grade', axis=1).sum()
    except KeyError:
        continue

    for gender in ['M', 'F']:
        try:
            row = subj_data.xs(gender, level='Gender').sum()
        except KeyError:
            continue
        total = row.sum()
        if total == 0:
            continue
        md_rate = (row.get('M', 0) + row.get('D', 0)) / total * 100
        perf_results.append({'Subject': subject, 'Gender': gender, 'MD_rate': round(md_rate, 1)})

perf_df = pd.DataFrame(perf_results).pivot(index='Subject', columns='Gender', values='MD_rate')
perf_df.columns = ['Female M+D%', 'Male M+D%']
perf_df['Female advantage'] = perf_df['Female M+D%'] - perf_df['Male M+D%']
perf_df = perf_df.sort_values('Female advantage', ascending=False)

print("Merit+Distinction rate by gender per subject:")
print(perf_df.to_string())

female_better = perf_df[perf_df['Female advantage'] > 0]
print(f"\nSubjects where females outperform males (higher M+D rate):")
for subj, row in female_better.iterrows():
    print(f"  {subj}: F={row['Female M+D%']:.1f}% vs M={row['Male M+D%']:.1f}% "
          f"(+{row['Female advantage']:.1f}pp advantage)")